## 1. Import knihoven

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.ticker import FuncFormatter

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)

## 2. Načtení dat

In [ ]:
# df = pd.read_csv("fotbal_prestupy_2000_2019.csv")

# df.shape
# df.head()
# df.info() - Odhadovaná hodnota  3440 non-null; Dtype jsou ruzne
# df.describe() - Vek min == 0
# list(df.columns)

dtype_mapper = {
    "Jméno": "string",
    "Pozice": "string",
    "Věk": "Int64",
    "Původní tým": "string",
    "Původní liga": "string",
    "Nový tým": "string",
    "Nová  Liga": "string", #Extra mezera
    "Sezóna": "string",
    "Odhadovaná hodnota": "Float64",
    "Přestupová částka": "Float64",
}
df = pd.read_csv("fotbal_prestupy_2000_2019.csv", dtype=dtype_mapper)

df.loc[df["Věk"] == 0, "Věk"] = pd.NA

df.rename(columns={"Nová  Liga": "Nová Liga"}, inplace=True)

df.info()


## 3. Transformace a čištění dat


**Odstranění potenciálních mezer**

In [ ]:
for column in list(df.columns):
    if df[column].dtype == "string":
        df[column] = df[column].str.strip()

df.info()

**Kontrola chybějících hodnot**

In [ ]:
pd.DataFrame({
    "pocet_null": df.isna().sum(),
    "podil_null": (df.isna().mean() * 100).round(2),
}).sort_values("pocet_null", ascending=False)

**Priprava dat pro vizualizace**

In [ ]:
df["Sezóna začátek"] = df["Sezóna"].str.extract(r"(\d{4})").astype("Int64")
df["V rámci stejné ligy"] = df["Původní liga"] == df["Nová Liga"]
df["Odhadovaná hodnota dostupná"] = df["Odhadovaná hodnota"].notna()
df["Poměr částka / hodnota"] = df["Přestupová částka"] / df["Odhadovaná hodnota"]
df["Rozdíl částka - hodnota"] = df["Přestupová částka"] - df["Odhadovaná hodnota"]
df["Věková skupina"] = pd.cut(
    df["Věk"],
    bins=[0, 20, 23, 26, 29, 40],
    labels=["<=20", "21-23", "24-26", "27-29", "30+"],
)

df.info()

## 4. Vizualizace

### 4.1 Vývoj trhu

In [ ]:
sezony = (
    df.groupby("Sezóna začátek")
    .agg(
        Pocet_prestupu=("Jméno", "count"),
        Celkem_castka=("Přestupová částka", "sum"),
        Median_castka=("Přestupová částka", "median"),
        Prumer_castka=("Přestupová částka", "mean"),
        Prumer_vek=("Věk", "mean"),
    )
    .reset_index()
)

sezony

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 6))
x_pozice = range(len(sezony))
x_popisky = sezony["Sezóna začátek"].astype(str)

ax1.bar(x_pozice, sezony["Celkem_castka"], color="#2a9d8f", alpha=0.85, label="Celková částka")
ax1.set_title("Vývoj trhu v čase")
ax1.set_xlabel("Začátek sezony")
ax1.set_ylabel("Celková částka přestupů")
ax1.set_xticks(list(x_pozice))
ax1.set_xticklabels(x_popisky, rotation=45)
ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x / 1_000_000_000:.1f} mld."))

ax2 = ax1.twinx()
ax2.plot(x_pozice, sezony["Prumer_castka"], marker="o", color="#e76f51", linewidth=2.5, label="Průměrná částka")
ax2.set_ylabel("Průměrná částka na transfer")
ax2.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x / 1_000_000:.1f} mil."))

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.show()

### 4.2 Kdo je kupec a kdo exportér

In [ ]:
bilance_lig = (
    pd.concat(
        [
            df.groupby("Nová Liga")["Přestupová částka"].sum().rename("Nakoupeno"),
            df.groupby("Původní liga")["Přestupová částka"].sum().rename("Prodano"),
        ],
        axis=1,
    )
    .fillna(0)
    .assign(
        Cista_bilance=lambda x: x["Nakoupeno"] - x["Prodano"],
        Celkovy_objem=lambda x: x["Nakoupeno"] + x["Prodano"],
    )
    .sort_values("Cista_bilance", ascending=False)
    .reset_index()
    .rename(columns={"index": "Liga"})
)

bilance_lig.head(10)

In [ ]:
graf_df = (
    bilance_lig.sort_values("Celkovy_objem", ascending=False)
    .head(10)
    .sort_values("Celkovy_objem", ascending=True)
)
graf_df_long = graf_df.melt(
    id_vars="Liga",
    value_vars=["Nakoupeno", "Prodano"],
    var_name="Typ toku",
    value_name="Částka",
)

plt.figure(figsize=(12, 7))
ax = sns.barplot(
    data=graf_df_long,
    y="Liga",
    x="Částka",
    hue="Typ toku",
    palette={"Nakoupeno": "#2a9d8f", "Prodano": "#e76f51"},
)
ax.set_title("Nákupy vs. prodeje podle lig (top 10 podle objemu)")
ax.set_xlabel("Přestupová částka")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x / 1_000_000_000:.1f} mld."))
ax.legend(title="")
plt.tight_layout()
plt.show()

### 4.3 Top 10 nakupujících a prodávajících klubů

In [ ]:
top_nakup = (
    df.groupby("Nový tým")["Přestupová částka"]
    .sum()
    .nlargest(10)
    .sort_values()
)
top_prodej = (
    df.groupby("Původní tým")["Přestupová částka"]
    .sum()
    .nlargest(10)
    .sort_values()
)

top_nakup_mil = top_nakup / 1_000_000
top_prodej_mil = top_prodej / 1_000_000

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(top_nakup_mil.index,  top_nakup_mil.values,  color="#2a9d8f")
axes[0].set_title('Top 10 kupujících klubů (2000/2018)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Celková útrata (mil. EUR)')
axes[1].barh(top_prodej_mil.index, top_prodej_mil.values, color="#e76f51")
axes[1].set_title('Top 10 prodávajících klubů (2000/2018)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Celkové příjmy (mil. EUR)')
plt.tight_layout()
plt.show()

### 4.4 Tržní hodnota vs. skutečná přestupová cena

In [ ]:
graf_df = df[df["Odhadovaná hodnota dostupná"]].copy()
graf_df["Odhadovaná hodnota (mil. EUR)"] = graf_df["Odhadovaná hodnota"] / 1_000_000
graf_df["Přestupová částka (mil. EUR)"] = graf_df["Přestupová částka"] / 1_000_000
# graf_df[graf_df["Poměr částka / hodnota"] > 4]["Poměr částka / hodnota"].describe()
graf_df["Poměr oříznutý"] = graf_df["Poměr částka / hodnota"].clip(upper=4) #Jinak se histogram rozbije kvůli pár extrémům

median_pomer = graf_df["Poměr částka / hodnota"].median()
max_osa = max(
    graf_df["Odhadovaná hodnota (mil. EUR)"].max(),
    graf_df["Přestupová částka (mil. EUR)"].max(),
)
print(f"Medián poměru částka / hodnota: {median_pomer:.2f}")
print(f"Průměr poměru částka / hodnota: {graf_df['Poměr částka / hodnota'].mean():.2f}")
print(f"Přestupů nad tržní hodnotu: {(len(graf_df[graf_df['Poměr částka / hodnota'] > 1])) / len(graf_df) * 100:.2f}%")

In [ ]:
graf_df[graf_df["Poměr částka / hodnota"] > 4]["Poměr částka / hodnota"].describe()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

sns.scatterplot(
    data=graf_df,
    x="Odhadovaná hodnota (mil. EUR)",
    y="Přestupová částka (mil. EUR)",
    alpha=0.25,
    s=22,
    color="#e78ac3",
    edgecolor=None,
    ax=ax1,
)
ax1.plot([0, max_osa], [0, max_osa], linestyle="--", color="red", label="Cena = hodnota")
ax1.set_title("Tržní hodnota vs. skutečná cena")
ax1.set_xlabel("Tržní hodnota (mil. EUR)")
ax1.set_ylabel("Přestupová částka (mil. EUR)")
ax1.legend()

sns.histplot(data=graf_df, x="Poměr oříznutý", bins=35, color="#6ad1c0", ax=ax2)
ax2.axvline(1, linestyle="--", color="red", label="Cena = hodnota")
ax2.axvline(median_pomer, color="#e76f51", label=f"Medián = {median_pomer:.2f}")
ax2.set_title("Histogram: cena / tržní hodnota")
ax2.set_xlabel("Poměr")
ax2.set_ylabel("Počet přestupů")
ax2.legend()

fig.suptitle("Vztah tržní hodnoty a přestupové ceny", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

### 4.5 Prémie podle věku

In [ ]:
premie_podle_veku = (
    df[df["Odhadovaná hodnota dostupná"]]
    .groupby("Věková skupina", observed=False)
    .agg(
        Pocet_prestupu=("Jméno", "count"),
        Median_pomer=("Poměr částka / hodnota", "median"),
        Median_rozdil=("Rozdíl částka - hodnota", "median"),
    )
    .reset_index()
)

premie_podle_veku

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=premie_podle_veku, x="Věková skupina", y="Median_pomer", color="#287271")
plt.title("Medián poměru částka / hodnota podle věku")
plt.xlabel("Věková skupina")
plt.ylabel("Medián poměru")
plt.tight_layout()
plt.show()

### 4.6 Distribuce přestupových částek podle pozice

In [ ]:
pocet_podle_pozice = df["Pozice"].value_counts()
vybrane_pozice = pocet_podle_pozice[pocet_podle_pozice >= 80].index

graf_df = df[df["Pozice"].isin(vybrane_pozice)].copy()
graf_df["Přestupová částka (mil. EUR)"] = graf_df["Přestupová částka"] / 1_000_000

poradi = (
    graf_df.groupby("Pozice")["Přestupová částka (mil. EUR)"]
    .median()
    .sort_values(ascending=False)
    .index
)

In [ ]:
plt.figure(figsize=(12, 7))
sns.boxplot(
    data=graf_df,
    y="Pozice",
    x="Přestupová částka (mil. EUR)",
    order=poradi,
    showfliers=False,
    color="#5aa9a2",
)
plt.title("Distribuce přestupových částek podle pozice")
plt.xlabel("Přestupová částka (mil. EUR)")
plt.ylabel("")
plt.tight_layout()
plt.show()

### 4.7 Heatmapa přestupů mezi top-5 ligami

In [ ]:
# Podle https://globalfootballrankings.com/
top_5_ligy = ["Premier League", "LaLiga", "Serie A", "1.Bundesliga", "Ligue 1"]

heatmap_top5 = (
    df[
        df["Původní liga"].isin(top_5_ligy)
        & df["Nová Liga"].isin(top_5_ligy)
    ]
    .groupby(["Původní liga", "Nová Liga"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=top_5_ligy, columns=top_5_ligy, fill_value=0)
)
heatmap_top5

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(
    heatmap_top5,
    annot=True,
    fmt="g",
    cmap="Blues",
    cbar_kws={"label": "Počet přestupů"},
)
plt.title("Počet přestupů mezi top-5 ligami")
plt.xlabel("Cílová liga")
plt.ylabel("Zdrojová liga")
plt.tight_layout()
plt.show()

## 5. Hlavní závěry

1. Trh po roce 2013 výrazně zdražuje; vrchol ve sledovaném období je sezóna 2017/2018.
2. Premier League je dominantní čistý kupec a zároveň největší centrum přestupového objemu.
3. Některé ligy a kluby fungují jako systematičtí exportéři talentu.
4. Největší prémii trh platí za mladé hráče, zejména do 20 let.
5. Nejdražší pozice: Left Winger > Right Winger > Central Midfield (dle mediánu)
6. Přibližně 63 % přestupů s dostupnou tržní hodnotou proběhlo nad odhadovanou hodnotou.

Důležitý limit: dataset nepopisuje celý trh, ale jen horní část trhu (top 250 transferů za sezonu).

## 6. Návrhy dalších hypotéz/analýzy
- **COVID-19 efekt (2019–20):** pokles cen po pandemii?
- **Vliv šampionátů (MS/ME):** rostou ceny o rok po turnaji?